# 03 — Preparar los ejemplos para aprender

Un pozo permite sacar fluidos del subsuelo. Nuestros registros cuentan petróleo, gas, agua y mediciones como horas, presión y temperatura. Queremos anticipar **petróleo futuro**, no recomendar cómo operar un pozo.

Cada ejemplo tendrá dos partes: las pistas que conocemos al terminar un día y la respuesta que sabremos siete días después. Este cuaderno las prepara; el 04 compara regresión y clasificación juntas.

Ejecutar antes el notebook 02 para generar la copia preparada. Las unidades del volumen diario siguen pendientes. Los siete días y el 20 % son una primera definición de trabajo; cambiarla requiere regenerar y evaluar todo, sin elegirla mirando la prueba final.

Las explicaciones debajo de los resultados describen la ejecución guardada de este experimento. Si cambian los datos o las decisiones y se vuelve a ejecutar, también hay que revisar esas lecturas.

## 1. Leer y comprobar de dónde vienen los datos

Usamos los cinco pozos que sólo registran producción. Verificamos que la copia todavía corresponde al Excel original. El código NPD es una identificación, como un nombre: lo leemos como texto. Conservamos la fila Excel para rastrear cada ejemplo.

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data" / "raw").is_dir(), "Abrir desde la raíz o notebooks/."
PROCESSED = ROOT / "data" / "processed"
fuente = PROCESSED / "produccion_exclusiva_preparada.csv"
assert fuente.is_file(), "Ejecutar primero el notebook 02."
reglas_02 = json.loads((PROCESSED / "reglas_preparacion.json").read_text(encoding="utf-8"))
raw = ROOT / "data" / "raw" / "Volve production data.xlsx"
assert hashlib.sha256(raw.read_bytes()).hexdigest() == reglas_02["sha256_fuente"], "La fuente cambió: regenerar el 02."
datos = pd.read_csv(fuente, parse_dates=["DATEPRD"], dtype={
    "NPD_WELL_BORE_CODE": "string", "NPD_FIELD_CODE": "string", "NPD_FACILITY_CODE": "string"
})
assert len(datos) == reglas_02["filas"]
assert sorted(datos["NPD_WELL_BORE_NAME"].unique()) == reglas_02["pozos"]
assert datos["FLOW_KIND"].eq("production").all()
assert not datos[["NPD_WELL_BORE_CODE", "DATEPRD"]].isna().any().any()
assert not datos.duplicated(["NPD_WELL_BORE_CODE", "DATEPRD"]).any()
datos = datos.sort_values(["NPD_WELL_BORE_CODE", "DATEPRD"]).reset_index(drop=True)
HORIZONTE = 7
CAIDA = 0.20
ID = "NPD_WELL_BORE_CODE"
FECHA = "DATEPRD"
PETROLEO = "BORE_OIL_VOL"
SEED = 42
print("Filas originales seleccionadas:", len(datos))

Filas originales seleccionadas: 9001


**Qué nos dejó esta celda.** Leímos 9.001 filas de cinco pozos y comprobamos sus identificaciones y el archivo de origen. Una fila es lo que se registró para un pozo en una fecha. Todavía no son 9.001 ejemplos útiles: falta revisar la calidad y encontrar su respuesta de siete días después. Que el archivo pase estos controles no garantiza que todas sus mediciones sean correctas.

## 2. Qué hacemos con los casos dudosos

Para este primer experimento **no usamos como día de partida ni como respuesta futura** registros marcados en el 02: horas fuera de 0–24, petróleo negativo, petróleo positivo con cero horas o producción con código WI. Es una decisión conservadora de modelado, no una afirmación de que esos registros estén mal. Guardamos los motivos. Una misma fila puede tener varias marcas.

No borramos los archivos de exploración. En las pistas históricas, el petróleo de un día marcado queda desconocido. Conservamos los días con cero horas y cero petróleo: representan una situación que el modelo necesita conocer. Los volúmenes negativos de gas/agua y valores no finitos no se usan como entradas; se dejan vacíos en esta copia. Los ceros de sensores se conservan porque su interpretación sigue pendiente.

Un vacío en las entradas se resolverá dentro de cada entrenamiento, usando sólo días anteriores. Nunca completamos la respuesta futura. En este experimento tampoco aceptamos un día de partida sin petróleo conocido/no negativo: lo necesitamos como referencia sencilla.

In [2]:
banderas = [c for c in datos if c.startswith("REVISAR_")]
assert banderas
assert all(pd.api.types.is_bool_dtype(datos[c]) for c in banderas), "Revisar lectura de banderas."
datos["DIA_DUDOSO"] = datos[banderas].any(axis=1)
numericas_fuente = ["ON_STREAM_HRS", "AVG_WHP_P", "DP_CHOKE_SIZE", "AVG_DOWNHOLE_PRESSURE",
                     "AVG_DOWNHOLE_TEMPERATURE", "AVG_DP_TUBING", "AVG_CHOKE_SIZE_P",
                     "AVG_WHT_P", "AVG_ANNULUS_PRESS", "BORE_GAS_VOL", "BORE_WAT_VOL"]
limpios = datos.copy()
for c in numericas_fuente + [PETROLEO]:
    limpios[c] = limpios[c].replace([np.inf, -np.inf], np.nan)
for c in [PETROLEO, "BORE_GAS_VOL", "BORE_WAT_VOL"]:
    limpios.loc[limpios[c].lt(0), c] = np.nan
limpios.loc[limpios["DIA_DUDOSO"], PETROLEO] = np.nan
limpios.loc[~limpios["AVG_CHOKE_UOM"].eq("%"), "AVG_CHOKE_SIZE_P"] = np.nan
limpios.loc[~limpios["ON_STREAM_HRS"].between(0, 24), "ON_STREAM_HRS"] = np.nan
display(datos.groupby("NPD_WELL_BORE_NAME")[banderas + ["DIA_DUDOSO"]].sum())

,REVISAR_HORAS,REVISAR_PETROLEO_NEGATIVO,REVISAR_PETROLEO_SIN_HORAS,REVISAR_TIPO_WI,DIA_DUDOSO
NPD_WELL_BORE_NAME,,,,,
15/9-F-1 C,1,0,0,2,3
15/9-F-11,2,0,1,0,3
15/9-F-12,5,0,0,0,5
15/9-F-14,5,0,0,0,5
15/9-F-15 D,0,0,0,0,0


**Cómo leer esta tabla.** Cada fila es un pozo y cada columna cuenta registros con una marca de revisión. REVISAR_HORAS cuenta horas fuera de 0–24; REVISAR_PETROLEO_SIN_HORAS, petróleo positivo con cero horas; REVISAR_TIPO_WI, producción con ese código pendiente de interpretar. DIA_DUDOSO cuenta días con al menos una marca, sin repetirlos.

En esta ejecución hay 16 días dudosos entre los cinco pozos: 13 por horas, uno por petróleo con cero horas y dos por tipo WI; no hay petróleo negativo marcado. Esos días no se usarán como partida o respuesta del experimento. No borramos sus filas de los archivos originales ni decimos que ya sabemos por qué ocurrió el problema.

## 3. Buscar el pasado y el futuro por fecha real

Si hoy fuera el 10 de octubre, ayer es el 9 y la respuesta buscada corresponde al 17. Si falta el 9, no usamos el 8 como si fuera ayer. Si falta el 17, la respuesta queda desconocida.

Las entradas son petróleo conocido hoy, hace uno y siete días; promedio y variación de los últimos siete días calendario, incluyendo hoy; cantidad de días conocidos; horas, sensores, gas y agua de hoy. Suponemos que el registro de hoy ya está disponible al emitir el pronóstico. Si llega con retraso, este planteo debe cambiar.

La clasificación pregunta si el petróleo de dentro de siete días es **como máximo el 80 % del de hoy**. Por ejemplo, pasar de 100 a 80 o menos es caída. Si hoy es cero no calculamos esa comparación porcentual: la fila puede servir para regresión, pero queda fuera de clasificación. Un cero futuro realmente observado sí puede ser caída.

In [3]:
indice_fuente = limpios.set_index([ID, FECHA])

def buscar_por_fecha(columna, dias):
    claves = pd.MultiIndex.from_arrays([limpios[ID], limpios[FECHA] + pd.Timedelta(days=dias)], names=[ID, FECHA])
    return pd.Series(indice_fuente[columna].reindex(claves).to_numpy(), index=limpios.index)

ejemplos = limpios[[ID, "NPD_WELL_BORE_NAME", FECHA, "FILA_EXCEL"]].copy()
ejemplos["FECHA_OBJETIVO"] = ejemplos[FECHA] + pd.Timedelta(days=HORIZONTE)
ejemplos["PETROLEO_T"] = limpios[PETROLEO]
ejemplos["PETROLEO_LAG_1D"] = buscar_por_fecha(PETROLEO, -1)
ejemplos["PETROLEO_LAG_7D"] = buscar_por_fecha(PETROLEO, -7)
for _, g in limpios.groupby(ID, sort=False):
    serie = g.set_index(FECHA)[PETROLEO]
    ventana = serie.rolling("7D", closed="right", min_periods=1)
    ejemplos.loc[g.index, "PETROLEO_MEDIA_7D"] = ventana.mean().to_numpy()
    ejemplos.loc[g.index, "PETROLEO_STD_7D"] = ventana.apply(lambda valores: np.nanstd(valores, ddof=0), raw=True).to_numpy()
    ejemplos.loc[g.index, "DIAS_CON_PETROLEO_7D"] = ventana.count().to_numpy()
ejemplos["PETROLEO_CAMBIO_7D"] = ejemplos["PETROLEO_T"] - ejemplos["PETROLEO_LAG_7D"]
for c in numericas_fuente:
    ejemplos[c] = limpios[c]
ejemplos["Y_PETROLEO_7D"] = buscar_por_fecha(PETROLEO, HORIZONTE)
ejemplos["FILA_EXCEL_OBJETIVO"] = buscar_por_fecha("FILA_EXCEL", HORIZONTE)
ejemplos["Y_CAIDA_20PCT"] = pd.Series(pd.NA, index=ejemplos.index, dtype="Int64")
clasificables = ejemplos["PETROLEO_T"].gt(0) & ejemplos["Y_PETROLEO_7D"].notna()
ejemplos.loc[clasificables, "Y_CAIDA_20PCT"] = (
    ejemplos.loc[clasificables, "Y_PETROLEO_7D"] <= (1 - CAIDA) * ejemplos.loc[clasificables, "PETROLEO_T"]
).astype(int)

auditoria = ejemplos[[ID, "NPD_WELL_BORE_NAME", FECHA, "FECHA_OBJETIVO", "FILA_EXCEL", "FILA_EXCEL_OBJETIVO"]].copy()
auditoria["PARTIDA_DUDOSA"] = datos["DIA_DUDOSO"]
auditoria["OBJETIVO_DUDOSO"] = buscar_por_fecha("DIA_DUDOSO", HORIZONTE).eq(True)
auditoria["SIN_FILA_OBJETIVO"] = ejemplos["FILA_EXCEL_OBJETIVO"].isna()
auditoria["SIN_PETROLEO_PARTIDA_VALIDO"] = ejemplos["PETROLEO_T"].isna()
auditoria["SIN_PETROLEO_OBJETIVO_VALIDO"] = ejemplos["Y_PETROLEO_7D"].isna()
auditoria["APTO_REGRESION"] = ~auditoria["PARTIDA_DUDOSA"] & ejemplos["PETROLEO_T"].notna() & ejemplos["Y_PETROLEO_7D"].notna()
auditoria["APTO_CLASIFICACION"] = auditoria["APTO_REGRESION"] & ejemplos["Y_CAIDA_20PCT"].notna()
ejemplos = ejemplos.loc[auditoria["APTO_REGRESION"]].copy()
display(auditoria.groupby("NPD_WELL_BORE_NAME")[["PARTIDA_DUDOSA", "OBJETIVO_DUDOSO", "SIN_FILA_OBJETIVO", "APTO_REGRESION", "APTO_CLASIFICACION"]].sum())

,PARTIDA_DUDOSA,OBJETIVO_DUDOSO,SIN_FILA_OBJETIVO,APTO_REGRESION,APTO_CLASIFICACION
NPD_WELL_BORE_NAME,,,,,
15/9-F-1 C,3,2,7,734,426
15/9-F-11,3,3,10,1149,1107
15/9-F-12,5,5,77,2969,2771
15/9-F-14,5,5,77,2969,2657
15/9-F-15 D,0,0,9,969,766


**Qué nos dejó esta celda.** Armamos parejas entre un día y su fecha exacta de siete días después. APTO_REGRESION cuenta parejas con petróleo válido en ambos extremos; APTO_CLASIFICACION cuenta las que además parten de petróleo positivo.

De las 9.001 filas iniciales quedan 8.790 ejemplos para estimar cantidades y 7.727 para clasificar caídas. La diferencia de 1.063 son partidas con petróleo cero: sirven para cantidades, pero no para una caída porcentual. Hay 180 filas sin registro en la fecha objetivo y 15 con objetivo dudoso. Los motivos de la tabla pueden coincidir: no se suman sus columnas como si todos fueran casos diferentes.

## 4. Separar aprender, practicar y rendir el examen

Tomamos el primer 75 % de las fechas disponibles para aprender, el siguiente 12,5 % para elegir entre alternativas y el último 12,5 % para la prueba final. Los límites se calculan con fechas, no con resultados del modelo, y son compartidos entre pozos.

Además apartamos del entrenamiento cualquier ejemplo cuya respuesta llegue al inicio de validación o después; hacemos lo mismo entre validación y prueba. Esos días quedan como separación. Así no aprendemos una respuesta que todavía no habríamos conocido al comenzar el período siguiente.

Los pozos tienen historias de distintas duraciones: las tablas muestran dónde participa cada uno. El cuaderno 02 ya exploró el historial completo; por eso esta prueba no es totalmente ajena a nuestra exploración previa. No cambiaremos los cortes ni las preguntas para mejorar el resultado del examen.

In [4]:
fechas = np.sort(ejemplos[FECHA].unique())
INICIO_VALIDACION = pd.Timestamp(fechas[int(len(fechas) * 0.75)])
INICIO_PRUEBA = pd.Timestamp(fechas[int(len(fechas) * 0.875)])
ejemplos["PARTICION"] = "Separación"
ejemplos.loc[(ejemplos[FECHA] < INICIO_VALIDACION) & (ejemplos["FECHA_OBJETIVO"] < INICIO_VALIDACION), "PARTICION"] = "Entrenamiento"
ejemplos.loc[(ejemplos[FECHA] >= INICIO_VALIDACION) & (ejemplos["FECHA_OBJETIVO"] < INICIO_PRUEBA), "PARTICION"] = "Validación"
ejemplos.loc[ejemplos[FECHA] >= INICIO_PRUEBA, "PARTICION"] = "Prueba"
assert ejemplos.loc[ejemplos["PARTICION"].eq("Entrenamiento"), "FECHA_OBJETIVO"].max() < INICIO_VALIDACION
assert ejemplos.loc[ejemplos["PARTICION"].eq("Validación"), "FECHA_OBJETIVO"].max() < INICIO_PRUEBA
assert (ejemplos["FECHA_OBJETIVO"] - ejemplos[FECHA]).eq(pd.Timedelta(days=7)).all()
assert ejemplos["DIAS_CON_PETROLEO_7D"].between(1, 7).all()
display(ejemplos.groupby("PARTICION").agg(Desde=(FECHA, "min"), Hasta=(FECHA, "max"), Ejemplos=(FECHA, "size")))
display(pd.crosstab(ejemplos["NPD_WELL_BORE_NAME"], ejemplos["PARTICION"]))
balance = ejemplos.dropna(subset=["Y_CAIDA_20PCT"]).groupby(["PARTICION", "NPD_WELL_BORE_NAME"])["Y_CAIDA_20PCT"].agg(["size", "sum", "mean"])
display(balance.rename(columns={"size": "Casos", "sum": "Caídas", "mean": "Proporción de caídas"}))

,Desde,Hasta,Ejemplos
PARTICION,,,
Entrenamiento,2008-02-12,2014-08-22,5199
Prueba,2015-09-05,2016-09-10,1711
Separación,2014-08-23,2015-09-04,70
Validación,2014-08-30,2015-08-28,1810


PARTICION,Entrenamiento,Prueba,Separación,Validación
NPD_WELL_BORE_NAME,,,,
15/9-F-1 C,135,223,14,362
15/9-F-11,403,372,14,360
15/9-F-12,2221,372,14,362
15/9-F-14,2221,372,14,362
15/9-F-15 D,219,372,14,364


Casos  Caídas  Proporción de caídas
PARTICION     NPD_WELL_BORE_NAME                                     
Entrenamiento 15/9-F-1 C            117      30               0.25641
              15/9-F-11             375      54                 0.144
              15/9-F-12            2095     315              0.150358
              15/9-F-14            1974     267              0.135258
              15/9-F-15 D           211      28              0.132701
Prueba        15/9-F-1 C            115      49              0.426087
              15/9-F-11             364      21              0.057692
              15/9-F-12             343      20              0.058309
              15/9-F-14             313      18              0.057508
              15/9-F-15 D           256      56               0.21875
Separación    15/9-F-1 C              1       0                   0.0
              15/9-F-11              10       1                   0.1
              15/9-F-12              10       0                   0.0
              15/9-F-14              10       0                   0.0
              15/9-F-15 D             5       0                   0.0
Validación    15/9-F-1 C            193      75              0.388601
              15/9-F-11             358      28              0.078212
              15/9-F-12             323      29              0.089783
              15/9-F-14             360      25              0.069444
              15/9-F-15 D           294      36              0.122449

**Cómo leer las tres tablas.** La primera muestra fechas y cantidad de ejemplos por etapa; la segunda, cuántos aporta cada pozo. En balance, Casos es cuántos ejemplos tienen clase, Caídas es cuántos cumplen la bajada del 20 % y Proporción es esa cantidad dividida por Casos: 0,10 significa diez de cada cien.

Quedan 5.199 ejemplos de cantidades para aprender, 1.810 para practicar y 1.711 para prueba. Otros 70 se apartan para que una respuesta futura no cruce al período siguiente. Validación empieza el 30/08/2014 y prueba el 05/09/2015. Los cinco pozos participan, aunque no aportan lo mismo.

Para clasificación son 4.772, 1.528 y 1.391 casos, respectivamente. Las caídas representan aproximadamente 14,5 %, 12,6 % y 11,8 %. Por eso acertar muchos días sin caída no basta para decir que el modelo es bueno.

## 5. Guardar reglas y ejemplos

Cada grupo agrega pistas al anterior. El nombre del pozo aparece en todos como categoría. Primero probamos petróleo pasado, después horas, presiones principales, otros sensores y finalmente gas/agua. La presión anular y temperatura en cabeza quedan fuera de esta primera comparación por lo visto en la exploración; no afirmamos que sean inútiles.

El CSV guarda también respuestas futuras para evaluar, pero **no son entradas**. El 04 usa una lista explícita de columnas permitidas para que ninguna respuesta se cuele en las pistas. El archivo de reglas guarda las fechas, supuestos y una huella de los datos: si cambian, hay que regenerar los resultados.

In [5]:
historia = ["PETROLEO_T", "PETROLEO_LAG_1D", "PETROLEO_LAG_7D", "PETROLEO_MEDIA_7D", "PETROLEO_STD_7D", "DIAS_CON_PETROLEO_7D", "PETROLEO_CAMBIO_7D"]
grupos = {
    "A_Historia": historia,
    "B_Horas": historia + ["ON_STREAM_HRS"],
    "C_Presiones": historia + ["ON_STREAM_HRS", "AVG_WHP_P", "DP_CHOKE_SIZE"],
    "D_Sensores": historia + ["ON_STREAM_HRS", "AVG_WHP_P", "DP_CHOKE_SIZE", "AVG_DOWNHOLE_PRESSURE", "AVG_DOWNHOLE_TEMPERATURE", "AVG_DP_TUBING", "AVG_CHOKE_SIZE_P"],
}
grupos["E_Gas_agua"] = grupos["D_Sensores"] + ["BORE_GAS_VOL", "BORE_WAT_VOL"]
salida = PROCESSED / "ejemplos_modelos_7d.csv"
ejemplos.to_csv(salida, index=False, date_format="%Y-%m-%d")
auditoria.to_csv(PROCESSED / "auditoria_ejemplos_7d.csv", index=False, date_format="%Y-%m-%d")
reglas = {
    "horizonte_dias": HORIZONTE, "umbral_caida": CAIDA, "momento": "Cierre de t; registros del día ya disponibles",
    "objetivo_regresion": "Petróleo del día calendario t+7, no acumulado semanal",
    "objetivo_clasificacion": "Petróleo(t)>0 y petróleo(t+7)<=0.8*petróleo(t); objetivo conocido/no negativo",
    "unidad_petroleo": "Pendiente de confirmar", "semilla": SEED,
    "fuente": fuente.relative_to(ROOT).as_posix(), "sha256_fuente": hashlib.sha256(fuente.read_bytes()).hexdigest(),
    "sha256_excel": reglas_02["sha256_fuente"], "sha256_ejemplos": hashlib.sha256(salida.read_bytes()).hexdigest(),
    "inicio_validacion": INICIO_VALIDACION.date().isoformat(), "inicio_prueba": INICIO_PRUEBA.date().isoformat(),
    "fracciones_fechas": [0.75, 0.125, 0.125], "purga": "Respuesta de una partición anterior estrictamente antes del comienzo de la siguiente",
    "grupos_numericos": grupos, "categoricas": ["NPD_WELL_BORE_NAME"],
    "calidad": "Excluir partida y objetivo marcados; petróleo histórico marcado a faltante; volúmenes negativos a faltante; sin imputar respuestas",
    "ventana": "Últimos 7 días calendario incluyendo t; media/std con al menos un dato y conteo explícito; std poblacional",
    "limitaciones": ["Sensores cero y unidades pendientes", "Registros supuestos disponibles al cierre del día", "Historial completo ya explorado; prueba no totalmente ajena al AED", "Clasificación excluye partidas sin petróleo positivo", "Sin inferencia causal ni intervenciones"],
}
(PROCESSED / "reglas_modelos_7d.json").write_text(json.dumps(reglas, ensure_ascii=False, indent=2), encoding="utf-8")
print("Guardado:", salida)
print("Siguiente: ejecutar 04_modelos.ipynb.")

Guardado: C:\Users\agusa\Desktop\ejercicios-facu\TP-FINAL_YPF\data\processed\ejemplos_modelos_7d.csv
Siguiente: ejecutar 04_modelos.ipynb.


**Qué quedó guardado.** El CSV de ejemplos contiene pistas y respuestas futuras; la auditoría explica qué filas sirven y cuáles no. El JSON es la receta: grupos de variables, fechas, reglas y huellas para comprobar los archivos.

Los grupos agregan información de a poco: A = petróleo pasado; B = A más horas; C = B más dos presiones; D = C más otros sensores; E = D más gas y agua. Esto permitirá preguntar si agregar pistas ayuda. Aquí todavía no entrenamos ni demostramos que alguna mejore las predicciones.